# 01 — RAG pipeline: load, clean, chunk, embed, retrieve

This notebook builds and tests the **retrieval** half of the RAG pipeline for `greek-hospitality-rag`:

```
load → clean → chunk → embed → retrieve (e5) → rerank (cross-encoder) → [generate: next notebook]
```

**How to run:** top to bottom (*Run All*). The first run downloads two models (~1 GB for e5, ~2.3 GB for the reranker). Embedding all chunks takes about 2 minutes on CPU.

Each section starts with a short explanation of *what* the cell does and *why* it was designed that way.

## 0. Setup

All imports, paths and settings in one place, so that changing an experiment parameter (e.g. chunk size or number of reranker candidates) means editing a single cell.

The notebook runs from the `notebooks/` folder, so paths start with `../`. Adding `..` to `sys.path` lets us import our own code from `src/`.

In [ ]:
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import CrossEncoder, SentenceTransformer

sys.path.insert(0, "..")
from src.eval_retrieval import evaluate_retrieval, load_questions, validate_phrases

# Paths
RAW_DIR = Path("../data/raw")
SOURCES_PATH = Path("../data/sources.csv")
QUESTIONS_PATH = Path("../eval/retrieval_questions.csv")

# Chunking
CHUNK_SIZE = 1000   # characters
OVERLAP = 200       # characters shared by neighbouring chunks

# Models
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

# Retrieval
TOP_K = 5           # chunks passed to the LLM
N_CANDIDATES = 30   # e5 candidates re-scored by the reranker

## 1. Load the documents page by page

Every PDF in `data/raw/` is read **one page at a time**, and each page keeps its `filename` and `page` number. Keeping the page number costs nothing now and lets the app later cite *"ΑΑΔΕ FAQ, p. 17"* instead of just the document name.

`sources.csv` holds the title, publisher, URL and date of each document. It is the single source of truth for document details, so chunks only store the `filename` and we look the rest up when we display a citation.

In [ ]:
sources = pd.read_csv(SOURCES_PATH)

pages = []
for pdf_path in sorted(RAW_DIR.glob("*.pdf")):
    reader = PdfReader(pdf_path)
    for page_num, page in enumerate(reader.pages, start=1):
        pages.append({
            "filename": pdf_path.name,
            "page": page_num,
            "text": page.extract_text() or "",
        })

print(len(pages), "pages from", len(sources), "documents")

## 2. Clean the text

PDF extraction leaves three kinds of noise that hurt search quality:

1. **TOC dot leaders**: tables of contents like `Βασικές Έννοιες ........ 2`. They are replaced by a space. This runs *before* step 3 so the spaces around the dots are collapsed too.
2. **Words split across lines** in ΦΕΚ PDFs: `βραχυχρό-\nνιας`, `κα - ταλυμάτων`. To the embedding model these are two meaningless fragments. We join them **only when both sides are lowercase Greek letters**, so real dashes such as `Α.1217/2023 - ΦΕΚ` are kept. `[ά-ώ]` covers all lowercase Greek letters (U+03AC–U+03CE), including accents and final ς.
3. **Whitespace**: any run of spaces, tabs or newlines becomes one space.

*Known limitation:* genuine hyphenated compounds (`οικονομικο-κοινωνικός`) are also merged, and words split in capitals (`ΒΡΑΧΥΧΡΟ- ΝΙΑΣ`) are not fixed. Both are rare in this corpus.

In [ ]:
def clean_text(text: str) -> str:
    # 1. Replace runs of dots (TOC leaders) with a space
    text = re.sub(r"\.{2,}", " ", text)
    # 2. Join words split by a hyphen, only between two lowercase Greek letters
    text = re.sub(r"([ά-ώ])\s*-\s*([ά-ώ])", r"\1\2", text)
    # 3. Collapse any run of spaces/newlines/tabs into one space
    text = re.sub(r"\s+", " ", text)
    return text.strip()

**Tests.** Each case documents one rule. If the regex changes later, these tell us immediately whether something that used to work is now broken.

In [ ]:
assert clean_text("βραχυχρό-\nνιας") == "βραχυχρόνιας"
assert clean_text("κα - ταλυμάτων") == "καταλυμάτων"
assert clean_text("τουριστι - κών") == "τουριστικών"
assert clean_text("Α.1217/2023 - ΦΕΚ") == "Α.1217/2023 - ΦΕΚ"      # real dash kept
assert clean_text("ΑΑΔΕ   \n\n  ΦΕΚ") == "ΑΑΔΕ ΦΕΚ"
assert clean_text("Βασικές Έννοιες........2") == "Βασικές Έννοιες 2"
assert clean_text("Βασικές Έννοιες ........ 2") == "Βασικές Έννοιες 2"  # spaces around dots
print("All clean_text tests passed")

Apply the cleaning to every page and look at the start of one ΦΕΚ page to check the result by eye.

In [ ]:
for p in pages:
    p["text"] = clean_text(p["text"])

law_page = next(p for p in pages if p["filename"] == "law_5170_2025_art3.pdf")
print(law_page["text"][:800])

## 3. Chunking

We search **chunks**, not whole pages, and each chunk gets its own embedding.

- **Size (1,000 characters):** large enough to keep context (*what* a fine is for), small enough that one embedding represents one topic. For Greek this is about 260–300 tokens, well under the model's 512-token limit (checked in section 4).
- **Overlap (200 characters):** if an answer falls on the boundary between two chunks, at least one of them still contains it whole.
- **Word boundaries:** a chunk never ends in the middle of a word.
- **Within each page:** every chunk has exactly one page number, which keeps citations simple. The trade-off is that a sentence crossing a page break is cut.

`max(..., start + 1)` guarantees the loop always moves forward, so no input can cause an infinite loop.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> list[str]:
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        # don't cut a word in half: move the end back to the last space
        if end < len(text):
            last_space = text.rfind(" ", start, end)
            if last_space > start:
                end = last_space
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        # next chunk starts `overlap` characters earlier, at a word boundary
        next_start = max(end - overlap, start + 1)
        space = text.find(" ", next_start, end)
        start = space + 1 if space != -1 else next_start
    return [c for c in chunks if c]

**Tests** on synthetic text (a different variable name, so the real `chunks` list is never overwritten).

In [ ]:
sample = " ".join(f"λέξη{i}" for i in range(2000))
test_chunks = chunk_text(sample)
assert all(len(c) <= CHUNK_SIZE for c in test_chunks)              # size limit respected
assert test_chunks[0].split()[-1] in test_chunks[1].split()        # overlap exists
assert chunk_text("") == []
assert chunk_text("σύντομο κείμενο") == ["σύντομο κείμενο"]
print("All chunk_text tests passed")

**Build the chunk list.** Each chunk is a dict with a unique `chunk_id` (`<filename>_p<page>_c<index>`), its `filename`, `page` and `text`. Title and URL are deliberately *not* copied in: they are looked up from `sources.csv` by `filename` when needed.

In [ ]:
chunks = []
for p in pages:
    for i, chunk in enumerate(chunk_text(p["text"])):
        chunks.append({
            "chunk_id": f"{p['filename']}_p{p['page']}_c{i}",
            "filename": p["filename"],
            "page": p["page"],
            "text": chunk,
        })

df_chunks = pd.DataFrame(chunks)
print(len(df_chunks), "chunks\n")
print(df_chunks["text"].str.len().describe(), "\n")
print(df_chunks.groupby("filename").size())

## 4. Embeddings

An embedding model turns a text into a vector (768 numbers for e5-base). Texts with similar **meaning** point in similar directions, even with no words in common.

**Why `multilingual-e5-base`:** good Greek support, runs on CPU, reads up to 512 tokens. (`paraphrase-multilingual-MiniLM-L12-v2` was ruled out: it reads only 128 tokens by default, so most of each chunk would be silently ignored.)

**e5 detail:** the model was trained with prefixes. Chunks are encoded as `"passage: ..."` and questions as `"query: ..."`. Without them, quality drops.

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)   # ~1 GB download the first time
print("Max tokens:", model.max_seq_length)

**Token check.** The longest chunk must be under 512 tokens, otherwise the model silently truncates it. Greek averages roughly 3.3 characters per token here.

In [ ]:
token_counts = [len(model.tokenizer("passage: " + c["text"])["input_ids"]) for c in chunks]
print(pd.Series(token_counts).describe())
assert max(token_counts) <= model.max_seq_length, "some chunks are truncated"

**Embed every chunk.** `normalize_embeddings=True` scales each vector to length 1, so cosine similarity becomes a simple dot product (this is also what the FAISS index will rely on). Takes about 2 minutes on CPU.

In [ ]:
passages = ["passage: " + c["text"] for c in chunks]
embeddings = model.encode(passages, normalize_embeddings=True,
                          batch_size=16, show_progress_bar=True)
print(embeddings.shape)   # (number of chunks, 768)

## 5. Retrieval with e5

`search_e5` embeds the question, computes the similarity to every chunk with one matrix product (`embeddings @ q`), and returns the chunk indices from most to least similar. Every search function in this notebook has the same signature, `(question, k) -> list of chunk indices`, so the evaluator can compare them directly.

In [ ]:
def search_e5(question: str, k: int) -> list[int]:
    q = model.encode("query: " + question, normalize_embeddings=True)
    scores = embeddings @ q
    return np.argsort(scores)[::-1][:k].tolist()


def show_top(question: str, search, k: int = TOP_K, preview: int = 200) -> None:
    """Print the top-k chunks for a question, for manual inspection."""
    for rank, idx in enumerate(search(question, k), start=1):
        c = chunks[idx]
        print(f"#{rank}  {c['filename']}  p.{c['page']}")
        print("   ", c["text"][:preview], "\n")

**Manual check** with one question (BKG-01). The correct answer is *"μέχρι τις 20 του επόμενου μήνα"*. Change the question to explore others.

In [ ]:
show_top("Μέχρι πότε πρέπει να δηλώσω μια μίσθωση στην ΑΑΔΕ;", search_e5)

## 6. Retrieval evaluation (baseline)

`eval/retrieval_questions.csv` has 24 answerable questions. Each has an **evidence phrase**: a regex (alternatives separated by `|`) that appears only in chunks that actually answer the question.

- **`hit@5`** (chunk level): any of the top 5 chunks contains the evidence phrase.
- **`doc_hit@5`** (document level): any of the top 5 chunks comes from an expected document. Easier, and it hides chunk-level misses, which is why we report both.
- **`first_rank`**: where the first correct chunk actually is (within the top 50).

`validate_phrases` first checks that every phrase still matches at least one chunk. Run it after any change to cleaning or chunking.

*Caution:* with 24 questions, one question ≈ 4 percentage points. Don't over-read small differences.

In [ ]:
questions = load_questions(QUESTIONS_PATH)
print("Phrases with no match:", validate_phrases(chunks, questions))

results_e5 = evaluate_retrieval(search_e5, chunks, questions, k=TOP_K)
print(results_e5[["hit@5", "doc_hit@5"]].mean())
results_e5[~results_e5["hit@5"]]

## 7. Reranking with a cross-encoder

**Why:** e5 is a *bi-encoder*: question and chunk are embedded **separately**. It is fast, but its scores are compressed (for BKG-01 the top 20 chunks were within 0.023 of each other), so the order inside the top 20 is close to noise.

A **cross-encoder** reads question and chunk **together** and outputs one relevance score, so it can tell *"mentions declarations"* apart from *"says when the declaration is due"*. It is too slow to run on every chunk, so we combine the two:

```
e5 → top 30 candidates (fast) → reranker re-scores the 30 → keep top 5
```

Details:
- The reranker gets the **plain question**: the `"query: "` prefix is specific to e5.
- `reranked + rest`: only the top 30 are re-scored; positions 31–50 keep their e5 order, so `first_rank` can still find chunks further down.

In [ ]:
reranker = CrossEncoder(RERANKER_MODEL, max_length=512)   # ~2.3 GB download the first time


def search_rerank(question: str, k: int, n_candidates: int = N_CANDIDATES) -> list[int]:
    ranked = search_e5(question, max(k, n_candidates))       # full e5 order
    candidates, rest = ranked[:n_candidates], ranked[n_candidates:]
    scores = reranker.predict([(question, chunks[i]["text"]) for i in candidates],
                              batch_size=8)
    reranked = [candidates[i] for i in np.argsort(scores)[::-1]]
    return (reranked + rest)[:k]

**Evaluate the reranker** on the same 24 questions. This takes a few minutes on CPU.

In [ ]:
results_rr = evaluate_retrieval(search_rerank, chunks, questions, k=TOP_K)
print(results_rr[["hit@5", "doc_hit@5"]].mean())
results_rr[~results_rr["hit@5"]]

### Your part: latency

Time `search_e5` and `search_rerank` on a few questions with `time.perf_counter()` and report the average seconds per query. On a free Hugging Face Space, this is how long users will wait.

In [ ]:
# TODO: measure average seconds per query for search_e5 and search_rerank

### Your part: which questions changed?

Compare `results_e5` and `results_rr` question by question. Which misses did the reranker fix? Did it break any question that e5 got right? (Hint: `results_e5.merge(results_rr, on="id", suffixes=("_e5", "_rr"))`.)

In [ ]:
# TODO: compare results_e5 and results_rr per question

## 8. Findings so far

| Variant | hit@5 | doc_hit@5 | Notes |
|---|---|---|---|
| Word overlap (reference) | 0.500 | 0.917 | naive shared-word count |
| e5-base, top 5 | 0.625 | 0.958 | baseline |
| e5 top 30 + bge-reranker-v2-m3 | ? | ? | |

- **Right document, wrong chunk:** `doc_hit@5` is almost perfect, while `hit@5` is not. A document-level metric alone would have hidden this.
- **Similarity scores don't separate answerable from unanswerable questions** (BKG-01 top score 0.879 vs. 0.823 for "Airbnb rules in Italy"), so refusal will be decided by the LLM prompt, with a score threshold only as a safety net.
- **The corpus contains superseded rules:** the 2024 guide gives the old ΤΑΚΚ amounts (1,50 € / 0,50 €) and ΠΟΛ 1187/2017 the old declaration deadline (next working day). The prompt must prefer the most recent source, using `published_date` from `sources.csv`.